# EDA: Processed NYC Yellow Taxi Data

This notebook explores `data/processed/yellow_tripdata_2024_processed.parquet`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_PATH = PROJECT_ROOT / 'data/processed/yellow_tripdata_2024_processed.parquet'


## Load Data

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
df.head()


## Dataset Overview

In [ ]:
overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(dropna=True),
})
overview.sort_values(['missing_pct', 'n_unique'], ascending=[False, False]).head(20)


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


## Quality Checks

In [ ]:
checks = {
    'negative_trip_distance': (df['trip_distance'] < 0).sum(),
    'negative_fare_amount': (df['fare_amount'] < 0).sum(),
    'negative_total_amount': (df['total_amount'] < 0).sum(),
    'negative_duration': (df['trip_duration_seconds'] < 0).sum(),
    'pickup_after_dropoff': (df['pickup_datetime'] > df['dropoff_datetime']).sum(),
}
pd.Series(checks).to_frame('count')


## Temporal Patterns

In [ ]:
hourly_trips = df.groupby('pickup_hour').size().rename('trip_count').reset_index()
plt.figure(figsize=(10, 4))
sns.lineplot(data=hourly_trips, x='pickup_hour', y='trip_count', marker='o')
plt.title('Trips by Pickup Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Trip Count')
plt.tight_layout()
plt.show()


In [ ]:
dow_trips = (
    df.groupby('pickup_dayofweek')
    .size()
    .rename('trip_count')
    .reset_index()
    .sort_values('pickup_dayofweek')
)
plt.figure(figsize=(10, 4))
sns.barplot(data=dow_trips, x='pickup_dayofweek', y='trip_count', palette='Blues_d')
plt.title('Trips by Day of Week (0=Mon)')
plt.xlabel('Pickup Day of Week')
plt.ylabel('Trip Count')
plt.tight_layout()
plt.show()


## Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(df['trip_distance'].clip(upper=30), bins=80, ax=axes[0], color='#1f77b4')
axes[0].set_title('Trip Distance Distribution (clipped at 30 miles)')

sns.histplot((df['trip_duration_seconds'] / 60).clip(upper=120), bins=80, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Trip Duration Distribution (minutes, clipped at 120)')

sns.histplot(df['total_amount'].clip(lower=0, upper=150), bins=80, ax=axes[2], color='#2ca02c')
axes[2].set_title('Total Amount Distribution (clipped at $150)')

for ax in axes:
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()


## Payment and Time-of-Day Mix

In [ ]:
payment_mix = (
    df['payment_type']
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
    .rename('pct')
    .to_frame()
)
payment_mix


In [ ]:
time_mix = (
    df['time_of_day']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename('pct')
    .reset_index()
    .rename(columns={'index': 'time_of_day'})
)
plt.figure(figsize=(8, 4))
sns.barplot(data=time_mix, x='time_of_day', y='pct', palette='viridis')
plt.title('Time-of-Day Share (%)')
plt.xlabel('Time of Day')
plt.ylabel('Share (%)')
plt.tight_layout()
plt.show()


## Top Zones (By ID)

In [ ]:
top_pickup = df['pickup_location_id'].value_counts().head(15)
top_dropoff = df['dropoff_location_id'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(x=top_pickup.values, y=top_pickup.index.astype(str), ax=axes[0], color='#4c72b0')
axes[0].set_title('Top 15 Pickup Location IDs')
axes[0].set_xlabel('Trip Count')
axes[0].set_ylabel('pickup_location_id')

sns.barplot(x=top_dropoff.values, y=top_dropoff.index.astype(str), ax=axes[1], color='#55a868')
axes[1].set_title('Top 15 Dropoff Location IDs')
axes[1].set_xlabel('Trip Count')
axes[1].set_ylabel('dropoff_location_id')

plt.tight_layout()
plt.show()


## Correlations

In [ ]:
corr_cols = [
    'trip_distance',
    'trip_duration_seconds',
    'passenger_count',
    'fare_amount',
    'tip_amount',
    'total_amount',
    'tolls_amount',
    'extra',
    'congestion_surcharge',
]

corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap (Selected Numeric Features)')
plt.tight_layout()
plt.show()

corr


## Key Takeaways

- Review demand concentration by hour/day to guide feature engineering for forecasting tasks.
- Validate outliers in distance, duration, and fare before modeling.
- Consider joining zone metadata (`data/external/taxi_zone_lookup.csv`) for richer spatial insights.
